In [1]:
import hashlib
import time
import json

class Block:
    def __init__(self, index, timestamp, data, previous_hash, nonce=0):
        self.index = index
        self.timestamp = timestamp
        self.data = data
        self.previous_hash = previous_hash
        self.nonce = nonce
        self.hash = self.compute_hash()

    def compute_hash(self):
        # We use json.dumps with sort_keys=True to ensure consistent ordering of attributes
        block_contents = json.dumps({
            "index": self.index,
            "timestamp": self.timestamp,
            "data": self.data,
            "previous_hash": self.previous_hash,
            "nonce": self.nonce
        }, sort_keys=True)
        return hashlib.sha256(block_contents.encode()).hexdigest()

    def mine_block(self, difficulty):
        """Helper method to find a valid hash that satisfies the Proof-of-Work."""
        target = "0" * difficulty
        while not self.hash.startswith(target):
            self.nonce += 1
            self.hash = self.compute_hash()

In [2]:
def verify_chain(chain, difficulty):
    if not chain:
        return True, "Chain is valid (Empty chain)."

    genesis_block = chain[0]
    if genesis_block.hash != genesis_block.compute_hash():
        return False, f"Block {genesis_block.index} [Hash Failure]: Genesis block hash has been tampered with."

    current_time = time.time()
    if genesis_block.timestamp > current_time:
        return False, f"Block {genesis_block.index} [Timestamp Failure]: Block is in the future."

    if len(chain) == 1:
        return True, "Chain is valid (Single Genesis block)."

    for i in range(1, len(chain)):
        current_block = chain[i]
        previous_block = chain[i - 1]

        if current_block.hash != current_block.compute_hash():
            return False, f"Block {current_block.index} [Hash Failure]: Stored hash does not match computed hash."

        if current_block.previous_hash != previous_block.hash:
            return False, f"Block {current_block.index} [Linkage Failure]: previous_hash does not match preceding block's actual hash."

        if not current_block.hash.startswith("0" * difficulty):
            return False, f"Block {current_block.index} [PoW Failure]: Hash does not meet difficulty requirements."

        if current_block.timestamp <= previous_block.timestamp:
            return False, f"Block {current_block.index} [Timestamp Failure]: Timestamp is older than or identical to previous block."
        if current_block.timestamp > time.time():
            return False, f"Block {current_block.index} [Timestamp Failure]: Block timestamp is in the future."

    return True, "Chain is valid. All integrity checks passed."

In [3]:
def run_tests():
    difficulty = 2 # Simple difficulty for fast testing

    print("--- Setting up a valid blockchain ---")
    genesis_block = Block(0, time.time() - 10, "Genesis Block", "0" * 64)
    genesis_block.mine_block(difficulty)

    chain = [genesis_block]

    for i in range(1, 3):
        time.sleep(0.1) # Ensure timestamp is strictly strictly greater
        new_block = Block(i, time.time(), f"Transaction Data {i}", chain[-1].hash)
        new_block.mine_block(difficulty)
        chain.append(new_block)

    is_valid, msg = verify_chain(chain, difficulty)
    print(f"Test 1 (Valid Chain): {msg}")

    import copy
    chain_tampered_data = copy.deepcopy(chain)
    chain_tampered_data[1].data = "Tampered Data!"
    is_valid, msg = verify_chain(chain_tampered_data, difficulty)
    print(f"Test 2 (Modified Data): {msg}")

    chain_tampered_linkage = copy.deepcopy(chain)
    chain_tampered_linkage[2].previous_hash = "11111111111111111111111111111111"
    chain_tampered_linkage[2].hash = chain_tampered_linkage[2].compute_hash()
    chain_tampered_linkage[2].mine_block(difficulty)
    is_valid, msg = verify_chain(chain_tampered_linkage, difficulty)
    print(f"Test 3 (Modified Linkage): {msg}")

    is_valid, msg = verify_chain([], difficulty)
    print(f"Test 4 (Empty Chain): {msg}")

    is_valid, msg = verify_chain([genesis_block], difficulty)
    print(f"Test 5 (Single Block Chain): {msg}")

    chain_multiple_tampered = copy.deepcopy(chain)
    chain_multiple_tampered[1].data = "Hacked Block 1" # Fails here first
    chain_multiple_tampered[2].timestamp = time.time() + 100000
    is_valid, msg = verify_chain(chain_multiple_tampered, difficulty)
    print(f"Test 6 (Multiple Tampered Blocks): {msg}")

if __name__ == "__main__":
    run_tests()

--- Setting up a valid blockchain ---
Test 1 (Valid Chain): Chain is valid. All integrity checks passed.
Test 2 (Modified Data): Block 1 [Hash Failure]: Stored hash does not match computed hash.
Test 3 (Modified Linkage): Block 2 [Linkage Failure]: previous_hash does not match preceding block's actual hash.
Test 4 (Empty Chain): Chain is valid (Empty chain).
Test 5 (Single Block Chain): Chain is valid (Single Genesis block).
Test 6 (Multiple Tampered Blocks): Block 1 [Hash Failure]: Stored hash does not match computed hash.
